In [1]:
import os, psutil
import time
import json
import pickle
import pandas as pd
import numpy as np
from functools import partial
from itertools import chain
import joblib

from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [2]:
import nltk
import networkx as nx
from collections import Counter
from text2graphapi.src.IntegratedSyntacticGraph import ISG

import torch

import optuna
import mlflow
from databricks.sdk import WorkspaceClient

from joblib import Parallel, delayed
import logging

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-22 12:44:58,865; - DEBUG; - Import libraries/modules from :PROD


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

True

Define path variables

In [5]:
representation_type = "integrated_syntactic_graph"
developer_initials = "JP"

In [6]:
current_dir = Path.cwd()
env_path = current_dir.parent.parent / "conf" / "local" / ".env"
results_path = current_dir.parent.parent / "results" / "graph"

train_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
validation_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-validation-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan21-authorship-verification-test-cleaned.jsonl"

train_data_isg_path = current_dir.parent.parent / "data" / "01_processed" / "training-vectors-isg.dat"
vocabulary_index_path = current_dir.parent.parent / "data" / "02_models" / "graph" / "vocab_index.pkl"

val_data_isg_path = current_dir.parent.parent / "data" / "01_processed" / "val-vectors-isg.dat"
test_data_isg_path = current_dir.parent.parent / "data" / "01_processed" / "test-vectors-isg.dat"

Connect to databricks for logging results

In [7]:
load_dotenv(env_path)

w = WorkspaceClient()   
print("Connected to:", w.config.host)

mlflow.set_tracking_uri("databricks")
mlflow.autolog()

2025/12/22 12:45:00 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.0 <= scikit-learn, but the installed version is 1.3.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2025/12/22 12:45:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/22 12:45:00 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.


Connected to: https://dbc-1ea3ad0e-f504.cloud.databricks.com


2025/12/22 12:45:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


What are GPU are the experiments run on

In [8]:
!nvidia-smi

Mon Dec 22 12:45:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     Off |   00000000:61:00.0 Off |                    0 |
|  0%   50C    P8             25W /  150W |       3MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
running_on_gpu = torch.cuda.is_available()

In [10]:
if running_on_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
else:
    gpu_name = "CPU"
    gpu_props = "N/A"
    gpu_vram_gb = 0

Empty the GPU from previous experiments

In [11]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

89

In [12]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NUMEXPR_MAX_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

Classification threshold constant specification

In [13]:
classification_thresholds = [x/1000 for x in range(200, 999)]

# Load dataset

#### Load training data

In [14]:
train_data_file_size = os.path.getsize(train_data_full_cleaned_path)
train_data = []

with open(train_data_full_cleaned_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

Loading data: 100%|██████████| 11.3G/11.3G [00:24<00:00, 457MB/s]


Successfully loaded 273301 items.


In [15]:
train_data_df = pd.DataFrame(train_data)

Prepare dataset for cosine embedding loss

In [16]:
train_data_df.head(10)

,id,pair,same
0,e05b9c0b-88a1-5608-b7e8-ab1fc6b78dc1,"[Well, ever since you and Kurt broke up youve ...",False
1,12f73a20-cdf3-58df-b5bb-9392eec9b486,"[The thing is, Ryouga has no reason to run aft...",False
2,d82c6764-451b-544c-8711-c139e9349c56,"[Ehhhh nah, its silly' Its my job to listen to...",True
3,876b8380-9260-5427-93e8-dc31155c3edd,"[Glaring at the arrogant spark, Always asks va...",False
4,357e8471-35b9-50b4-9ac0-286ac0e8b101,[Runa limped across the small space to an open...,False
5,6b39fe22-409f-5329-9fe1-ddf4a70bedd1,"[And thats retired Commander, if you please St...",False
6,76ed2017-0c9f-580f-83b1-5a041a8169ec,[Meet you downstairs in twenty minutes I say w...,True
7,fd8deb2e-06de-5c92-9a4c-927891c53657,"[Since Ive seen so many others do so, Im going...",False
8,3ce5e811-a57c-5fbf-9f9a-2ee636e50be6,[party Eishi exclaimed Omi shook his head and ...,False
9,25a17cd2-6b01-5fba-99ef-e631e56e181d,[After a few moments she found that Red was ri...,False


#### Load validation data

In [17]:
val_data_file_size = os.path.getsize(validation_data_full_cleaned_path)
val_data = []

with open(validation_data_full_cleaned_path, 'r') as f:
    with tqdm(total=val_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            val_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(val_data)} items.")

Loading data: 100%|██████████| 103M/103M [00:00<00:00, 508MB/s] 


Successfully loaded 2500 items.


In [18]:
val_data_df = pd.DataFrame(val_data)

#### Load testing data

In [19]:
test_data_file_size = os.path.getsize(test_data_full_cleaned_path)
test_data = []

with open(test_data_full_cleaned_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 826M/826M [00:02<00:00, 348MB/s] 


Successfully loaded 19999 items.


In [20]:
test_data_df = pd.DataFrame(test_data)

# Functions to build graphs and extract features

In [21]:
def texts_to_isg_graphs(texts, n_jobs=-1):
    def process(id, text):
        
        logging.disable(logging.INFO)
        logging.getLogger('text2graphapi').setLevel(logging.WARNING)
        logging.getLogger('text2graphapi.models').setLevel(logging.WARNING)
        
        isg = ISG(
            graph_type="DiGraph",
            language="en",
            apply_prep=True,
            output_format="networkx"
        )
        corpus = [{"id": id, "doc": text}]
        graph_object = isg.transform(corpus)[0]["graph"]
        return graph_object

    graphs = Parallel(n_jobs=n_jobs)(
        delayed(process)(id, text)
        for id, text in tqdm(
            enumerate(texts),
            total=len(texts),
            desc="Processing ISGs, print_"
        )
    )

    return graphs

Parse ISG's nodes POS and lemma function

In [22]:
def parse_graph_node(graph_node):
    node_str = str(graph_node)
    if "_" in node_str:
        lemma, pos = node_str.rsplit("_", 1)
        return lemma.lower(), pos

Parse dependency

In [23]:
def parse_graph_dependency(data):
    dependency = data.get("gramm_relation")
    parsed_dependency = dependency.split("_", 1)[0]
    return parsed_dependency

Extract multi-level features from graph

In [24]:
def extract_features_from_isg(graph):
    features = Counter()

    for node in graph.nodes:
        lemma, pos = parse_graph_node(node)
        features[f"LEX::{lemma}"] += 1
        
        if pos:
            features[f"POS::{pos}"] += 1

    for _, _, data in graph.edges(data=True):
        dependency = parse_graph_dependency(data)
        if dependency:
            features[f"DEP::{dependency}"] += 1

    return features

Build vocabulary from counters function

In [25]:
def build_vocab(counters):
    vocab = sorted(set().union(*counters))
    index = {f: i for i, f in enumerate(vocab)}
    return vocab, index

Build vectors based on vocabulary

In [26]:
def build_vector(features, index):
    vector = np.zeros(len(index), dtype=np.float32)
    for feature, value in features.items():
        if feature in index:
            vector[index[feature]] = value
    return vector

Print process RAM usage

In [27]:
def print_ram_usage():
    print(f"Process RAM usage: {process.memory_info().rss / 1e9:.2f} GB")

Convert texts to text2graphapi integrated syntactic graphs

In [28]:
def convert_train_texts_to_vectors(input_df, n_jobs=1):
    print_ram_usage()
    texts1 = input_df["pair"].apply(lambda x: x[0])
    texts2 = input_df["pair"].apply(lambda x: x[1])
    print_ram_usage()
    X1 = texts_to_isg_graphs(texts1, n_jobs)
    del texts1
    print_ram_usage()
    X2 = texts_to_isg_graphs(texts2, n_jobs)
    del texts2
    print_ram_usage()
    
    print("Extracting features from the graph \n")
    features1 = [extract_features_from_isg(graph) for graph in tqdm(X1, desc="extracting freatures")]
    del X1
    print_ram_usage()
    
    features2 = [extract_features_from_isg(graph) for graph in tqdm(X2, desc="extracting freatures")]
    del X2
    print_ram_usage()
        
    print("Building vocab \n")
    vocab, index = build_vocab(chain(features1, features2))
    del vocab
    print_ram_usage()
    
    print("Building vectors \n")
    vectors1 = [
        build_vector(features, index)
        for features in tqdm(features1, desc="Building vectors - first of the pair")
    ]
    
    del features1
    print_ram_usage()
    
    vectors2 = [
        build_vector(features, index)
        for features in tqdm(features2, desc="Building vectors - second of the pair")
    ]
    del features2
    print_ram_usage()
    
    return index, vectors1, vectors2

Convert test texts to vectors

In [29]:
def convert_test_texts_to_vectors(input_df, index, n_jobs=1):
        
    texts1 = input_df["pair"].apply(lambda x: x[0])
    texts2 = input_df["pair"].apply(lambda x: x[1])

    X1 = texts_to_isg_graphs(texts1, n_jobs)
    del texts1
    
    X2 = texts_to_isg_graphs(texts2, n_jobs)
    del texts2
    print_ram_usage()
    
    print("Extracting features from the graph \n")
    features1 = [extract_features_from_isg(graph) for graph in tqdm(X1, desc="extracting freatures")]
    del X1
    
    features2 = [extract_features_from_isg(graph) for graph in tqdm(X2, desc="extracting freatures")]
    del X2
    print_ram_usage()
    
    print("Building vectors \n")
    vectors1 = [
        build_vector(features, index)
        for features in tqdm(features1, desc="Building vectors - first of the pair")
    ]
    del features1
    
    vectors2 = [
        build_vector(features, index)
        for features in tqdm(features2, desc="Building vectors - second of the pair")
    ]
    del features2
    print_ram_usage()    
    
    return vectors1, vectors2

# Build graphs, get features for training data and get the vocabulary index

In [ ]:
process = psutil.Process(os.getpid())
index, train_vectors1, train_vectors2 = convert_train_texts_to_vectors(train_data_df, n_jobs=8)

Process RAM usage: 13.24 GB
Process RAM usage: 13.25 GB


Processing ISGs, print_:   0%|          | 0/273301 [00:00<?, ?it/s][nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data] 

2025-12-22 12:45:32,934; - DEBUG; - Import libraries/modules from :PROD
2025-12-22 12:45:32,938; - DEBUG; - Import libraries/modules from :PROD
2025-12-22 12:45:32,951; - DEBUG; - Import libraries/modules from :PROD
2025-12-22 12:45:33,005; - DEBUG; - Import libraries/modules from :PROD
2025-12-22 12:45:33,039; - DEBUG; - Import libraries/modules from :PROD
2025-12-22 12:45:33,109; - DEBUG; - Import libraries/modules from :PROD
2025-12-22 12:45:33,159; - DEBUG; - Import libraries/modules from :PROD
2025-12-22 12:45:33,207; - DEBUG; - Import libraries/modules from :PROD


Processing ISGs, print_:   0%|          | 208/273301 [00:29<9:24:07,  8.07it/s] [nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Processing ISGs, print_:   0%|          | 216/273301 [00:30<9:58:28,  7.61it/s]

2025-12-22 12:46:00,028; - DEBUG; - Import libraries/modules from :PROD


...

Processing ISGs, print_:  26%|██▌       | 70600/273301 [3:20:47<15:37:13,  3.60it/s][nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Processing ISGs, print_:  26%|██▌       | 70608/273301 [3:20:48<13:49:53,  4.07it/s]

2025-12-22 16:06:18,564; - DEBUG; - Import libraries/modules from :PROD


Processing ISGs, print_:  26%|██▌       | 70624/273301 [3:20:52<13:13:07,  4.26it/s][nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-22 16:06:23,744; - DEBUG; - Import libraries/modules from :PROD


Processing ISGs, print_:  26%|██▌       | 70632/273301 [3:20:55<15:04:55,  3.73it/s]

In [ ]:
train_data_length = len(train_data_df)
dimension = len(train_vectors1[0])

train_embeddings = np.memmap(
    train_data_isg_path,
    dtype="float32",
    mode="w+",
    shape=(train_data_length, 2, dimension)
)


train_vectors1 = np.asarray(train_vectors1, dtype="float32")
train_vectors2 = np.asarray(train_vectors2, dtype="float32")

train_embeddings[:, 0, :] = train_vectors1 
train_embeddings[:, 1, :] = train_vectors2

In [ ]:
del train_embeddings,  train_vectors1,  train_vectors2

In [ ]:
with open(vocabulary_index_path, "wb") as f:
    pickle.dump(index, f)

# Create vectors for testing and validation data and save them

In [ ]:
index = None
with open(vocabulary_index_path, "rb") as f:
    index = pickle.load(f)

Convert validation data to vectors

In [ ]:
val_vectors1, val_vectors2 = convert_test_texts_to_vectors(val_data_df, index, n_jobs=8)

In [ ]:
val_data_length = len(val_data_df)

val_embeddings = np.memmap(
    val_data_isg_path,
    dtype="float32",
    mode="w+",
    shape=(val_data_length, 2, dimension)
)

val_vectors1 = np.asarray(val_vectors1, dtype="float32")
val_vectors2 = np.asarray(val_vectors2, dtype="float32")

val_embeddings[:, 0, :] = val_vectors1 
val_embeddings[:, 1, :] = val_vectors2

In [ ]:
del val_embeddings,  val_vectors1,  val_vectors2

Convert test data to vectors

In [ ]:
test_vectors1, test_vectors2 = convert_test_texts_to_vectors(test_data_df, index, n_jobs=8)

In [ ]:
test_data_length = len(test_data_df)

test_embeddings = np.memmap(
    test_data_isg_path,
    dtype="float32",
    mode="w+",
    shape=(test_data_length, 2, dimension)
)

test_vectors1 = np.asarray(test_vectors1, dtype="float32")
test_vectors2 = np.asarray(test_vectors2, dtype="float32")

test_embeddings[:, 0, :] = test_vectors1 
test_embeddings[:, 1, :] = test_vectors2

In [ ]:
del test_embeddings,  test_vectors1,  test_vectors2